# 06 Hypergraph Extension

Notebook này minh họa cách dựng `group_context` từ current patient + top-K neighbors bằng hypergraph.

In [ ]:
from pathlib import Path
import json
import pyarrow.parquet as pq

from src.data.dataset import collate_batch
from src.models.patient_state_encoder import PatientStateEncoder
from src.graph.group_encoder import GroupEncoder
from src.retrieval.memory_bank import MemoryBank, build_last_visit_queries
from src.retrieval.topk_retriever import retrieve_topk


In [ ]:
root = Path.cwd().resolve().parents[0]
handover_root = root / 'handover_data'
records = pq.read_table(handover_root / 'processed' / 'train' / 'part-00000.parquet').slice(0, 4).to_pylist()
with (handover_root / 'vocab' / 'diagnosis_vocab.json').open('r', encoding='utf-8') as handle:
    diagnosis_vocab_size = json.load(handle)['size']
with (handover_root / 'vocab' / 'procedure_vocab.json').open('r', encoding='utf-8') as handle:
    procedure_vocab_size = json.load(handle)['size']
with (handover_root / 'vocab' / 'drug_vocab.json').open('r', encoding='utf-8') as handle:
    drug_vocab_size = json.load(handle)['size']

batch = collate_batch(records)
encoder = PatientStateEncoder(
    diagnosis_vocab_size=diagnosis_vocab_size,
    procedure_vocab_size=procedure_vocab_size,
    drug_vocab_size=drug_vocab_size,
    num_lab_features=batch['lab_values'].shape[-1],
    num_vital_features=batch['vital_values'].shape[-1],
    code_embedding_dim=8,
    medication_embedding_dim=8,
    numeric_projection_dim=4,
    time_embedding_dim=4,
    visit_hidden_dim=16,
    hidden_dim=12,
    dropout=0.0,
)
group_encoder = GroupEncoder(hidden_dim=12, num_layers=2, dropout=0.0, num_group_prototypes=4)

encoder_outputs = encoder(batch)
bank = MemoryBank.build_from_batch(records, encoder_outputs, split='train')
query_states, query_metadata = build_last_visit_queries(records, encoder_outputs, split='train')
retrieval_payload = retrieve_topk(query_states, query_metadata, bank, top_k=2, temporal_decay_alpha=0.05)
group_outputs = group_encoder(
    current_state=encoder_outputs['pooled_state'],
    retrieval_payload=retrieval_payload,
    memory_bank=bank,
)

group_outputs['group_context'].shape, group_outputs['cluster_label']
